In [22]:
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings("ignore")

#### Importing the dataset comprising data of children with illegal immigration status or no parent or guardian

In [15]:
df = pd.read_csv("HHS_Unaccompanied_Alien_Children_Program.csv")
df.head()

,Date,Children apprehended and placed in CBP custody*,Children in CBP custody,Children transferred out of CBP custody,Children in HHS Care,Children discharged from HHS Care
0,21-Dec-25,6,18,11,"2,484",14
1,18-Dec-25,11,50,6,"2,472",16
2,17-Dec-25,7,31,11,"2,481",10
3,16-Dec-25,8,54,15,"2,468",9
4,15-Dec-25,11,42,9,"2,470",7


#### Checking for null values

In [16]:
df.isna().sum()

Date                                               0
Children apprehended and placed in CBP custody*    0
Children in CBP custody                            0
Children transferred out of CBP custody            0
Children in HHS Care                               0
Children discharged from HHS Care                  0
dtype: int64

#### Converting data to pandas date-time format

In [17]:
df["Date"] = pd.to_datetime(df["Date"])
df.head()

,Date,Children apprehended and placed in CBP custody*,Children in CBP custody,Children transferred out of CBP custody,Children in HHS Care,Children discharged from HHS Care
0,2025-12-21,6,18,11,"2,484",14
1,2025-12-18,11,50,6,"2,472",16
2,2025-12-17,7,31,11,"2,481",10
3,2025-12-16,8,54,15,"2,468",9
4,2025-12-15,11,42,9,"2,470",7


#### Arrange dates in ascending order

In [18]:
df.sort_values(by=["Date"], inplace=True)
df.reset_index(drop=True, inplace=True)
df.head()

,Date,Children apprehended and placed in CBP custody*,Children in CBP custody,Children transferred out of CBP custody,Children in HHS Care,Children discharged from HHS Care
0,2023-01-12,33,53,34,"6,566",436
1,2023-01-22,32,49,39,"7,122",227
2,2023-01-23,32,50,39,"7,280",181
3,2023-01-24,47,42,47,"7,433",175
4,2023-01-25,20,22,41,"7,538",180


#### Retrieving weekday names for each date in the dataset

In [19]:
df["Day of Week"] = list(map(lambda x: x.strftime("%A"), df["Date"]))
df = df[["Date","Day of Week","Children apprehended and placed in CBP custody*","Children in CBP custody","Children transferred out of CBP custody","Children in HHS Care","Children discharged from HHS Care"]]
df.head()

,Date,Day of Week,Children apprehended and placed in CBP custody*,Children in CBP custody,Children transferred out of CBP custody,Children in HHS Care,Children discharged from HHS Care
0,2023-01-12,Thursday,33,53,34,"6,566",436
1,2023-01-22,Sunday,32,49,39,"7,122",227
2,2023-01-23,Monday,32,50,39,"7,280",181
3,2023-01-24,Tuesday,47,42,47,"7,433",175
4,2023-01-25,Wednesday,20,22,41,"7,538",180


#### Finding out the total number of records on each day of week

In [20]:
weekday_number = df["Day of Week"].value_counts().reset_index()
weekday_number["Day of Week Number"] = weekday_number["Day of Week"].map({
    "Sunday": "1",
    "Monday": "2",
    "Tuesday": "3",
    "Wednesday": "4",
    "Thursday": "5",
    "Friday": "6",
    "Saturday": "7"
})
weekday_number = weekday_number.sort_values(by=["Day of Week Number"]).reset_index(drop=True)
weekday_number

,Day of Week,count,Day of Week Number
0,Sunday,130,1
1,Monday,145,2
2,Tuesday,149,3
3,Wednesday,147,4
4,Thursday,147,5
5,Friday,2,6


Looking at the numbers above, we can see that the number of days on which children were recorded is relatively consistent from Sunday through Thursday. From Monday through Thursday, the figures range between 145 and 150 days. Sunday is somewhat lower, at 130 days, but it is still reasonably close to this range.

Friday, however, stands out as an exceptionally low number of recorded days. Data is recorded for only two Fridays, compared with more than 130 recorded days for each of the other weekdays. This substantial difference suggests that Friday may represent a distinct pattern in the data and warrants further investigation.

There is one assumption that we can make here, which is deleting those two Friday records and assuming that services are not provided on Fridays and Saturdays. But in reality, these services are provided 24 hours a day and 7 days a week. So we need to include these days on which data is not recorded and then somehow deal with the null values for number of children on these days.

#### Creating a new dataset with dates on which  data is not recorded

In [21]:
df_2 = pd.DataFrame(columns=df.columns)
df_2["Date"] = np.setdiff1d(pd.date_range(df["Date"].min(),df["Date"].max()).values,df["Date"].values)
df_2["Day of Week"] = list(map(lambda x: x.strftime("%A"), df_2["Date"]))
df_2[['Children apprehended and placed in CBP custody*','Children in CBP custody','Children transferred out of CBP custody','Children in HHS Care', 'Children discharged from HHS Care']] = np.nan
df_2.head()

,Date,Day of Week,Children apprehended and placed in CBP custody*,Children in CBP custody,Children transferred out of CBP custody,Children in HHS Care,Children discharged from HHS Care
0,2023-01-13,Friday,NaN,NaN,NaN,NaN,NaN
1,2023-01-14,Saturday,NaN,NaN,NaN,NaN,NaN
2,2023-01-15,Sunday,NaN,NaN,NaN,NaN,NaN
3,2023-01-16,Monday,NaN,NaN,NaN,NaN,NaN
4,2023-01-17,Tuesday,NaN,NaN,NaN,NaN,NaN


#### Concatenating the original dataset with the new dataset comprising unrecorded dates

In [9]:
df_3 = pd.concat([df,df_2]).sort_values(by=["Date"]).reset_index(drop=True)
df_3.head()

,Date,Day of Week,Children apprehended and placed in CBP custody*,Children in CBP custody,Children transferred out of CBP custody,Children in HHS Care,Children discharged from HHS Care
0,2023-01-12,Thursday,33.0,53.0,34.0,"6,566",436.0
1,2023-01-13,Friday,NaN,NaN,NaN,NaN,NaN
2,2023-01-14,Saturday,NaN,NaN,NaN,NaN,NaN
3,2023-01-15,Sunday,NaN,NaN,NaN,NaN,NaN
4,2023-01-16,Monday,NaN,NaN,NaN,NaN,NaN
